# Agentic AI for Science, Part 2: An Agent Hunting for Cancer Drug Targets

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nathanmaulding/tutorial-projects/blob/main/agentic-ai-for-science/03_agentic_target_discovery.ipynb)

Part 1 tested an agent where the right answers were known. Here we point the same framework at an **open question with real data**: across about 1,000 cancer cell lines in which every gene was knocked out one at a time with CRISPR, **which genes do cancers with a specific alteration depend on, that other cancers don't?** Each such pair (a *biomarker* and a *selective dependency*) is a candidate drug target that comes with a way to choose patients. This is target identification and validation, done computationally.

The hard part isn't finding associations: a genome-wide scan produces thousands. It's knowing **which ones are real, which are artifacts, and which are new.** That's where the agent's judgment, and the guard rails around it, are tested.

**What you'll learn**

1. Why discovery statistics alone mislead, even with strict multiple-testing correction
2. Designing guard rails into an agent's tools: **pre-registration, a replication budget, a fixed threshold**
3. **Independent replication**: held-out cell lines, plus a different lab's screens with a different CRISPR library
4. **Literature retrieval** as a novelty check, and how it can fool you
5. **Null worlds**: running the whole agent on data with every true signal destroyed, to measure its false-discovery rate
6. What the agents found, ranked, with the caveats a reviewer would raise

**Runtime:** seconds; this notebook reads recorded runs. Re-running the agents needs the DepMap data (about 1.6 GB, see section 9) and your own API key.

> 🎬 *Video walkthrough: coming soon.*

## 0. Setup

In [1]:
import os, sys, json
if "google.colab" in sys.modules and not os.path.exists("agentlab"):
    !git clone -q --depth 1 https://github.com/nathanmaulding/tutorial-projects.git
    %cd tutorial-projects/agentic-ai-for-science
    !pip install -q -r requirements.txt

import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from target_discovery import analysis as A

pd.set_option("display.max_colwidth", 80)
df = A.load()
print(f"{len(df)} recorded sessions; agent API spend for Part 2: ${df.usd.fillna(0).sum():.2f}")

61 recorded sessions; agent API spend for Part 2: $5.51


## 1. The data, and the replication design

[DepMap](https://depmap.org) (the Broad Institute's Cancer Dependency Map) knocks out each of roughly 18,000 genes in each of about 1,000 cancer cell lines and measures the effect on growth. A **gene effect** of 0 means no effect and −1 is a typical essential gene; more negative means the cells depend on that gene more. We pair that with each cell line's **biomarkers**:

| Biomarker | Meaning |
|---|---|
| `MUT_<gene>` | damaging (likely loss-of-function) mutation |
| `HOT_<gene>` | hotspot (likely activating) mutation |
| `LOSS_<gene>` | expression lost in a normally expressed gene (deletion or silencing) |
| `LIN_<name>` | cancer lineage |

**The replication design is the core of this project.** The data are split three ways *before* the agent sees anything:

In [2]:
s = json.load(open("runs/part2/data_summary.json"))
pd.DataFrame({
    "cell lines": [s["discovery_lines"], s["holdout_lines"], s["sanger_lines"]],
    "screens": ["Broad (Avana library)", "Broad (Avana library)", "Sanger Project Score (KY library)"],
    "who sees it": ["the agent, freely", "only via replicate()", "only via replicate()"],
}, index=["discovery", "held-out", "independent lab"])

,cell lines,screens,who sees it
discovery,626,Broad (Avana library),"the agent, freely"
held-out,417,Broad (Avana library),only via replicate()
independent lab,315,Sanger Project Score (KY library),only via replicate()


- **Discovery:** 60% of Broad's cell lines (split within each cancer lineage). This is all the agent can explore.
- **Held-out:** the other 40% of Broad's lines. A new sample of cell lines from the same experimental pipeline.
- **Independent lab:** the Sanger Institute's Project Score screens, run with a different CRISPR library and processed separately. A finding that replicates here survived a change of lab, reagents and pipeline.

All tests are **lineage-adjusted**: gene effect is regressed on the biomarker plus cancer-lineage indicators, so "both are common in melanoma" can't masquerade as biology.

## 2. The guard rails are in the tools, not just the prompt

An agent can be told to be rigorous. It's more reliable to make rigor *the only path the tools allow*:

| Guard rail | How it's enforced |
|---|---|
| The agent never sees replication data | held-out and Sanger data are reachable only through `replicate()` |
| **Pre-registration** | `replicate()` refuses any hypothesis not registered first; registration freezes the biomarker, target, direction, proposed mechanism and novelty assessment, and can't be edited |
| **Fixed budget** | 5 replications per session |
| **Fixed threshold, chosen in advance** | one-sided p < 0.01 in *both* held-out and Sanger (0.05 Bonferroni-corrected for the 5 tests) |
| **Honest reporting** | the final report is checked: a finding marked "replicated" must match what `replicate()` returned |

This mirrors pre-registration in clinical trials and in good computational biology: decide the test before you see the answer, and limit how many shots you take.

## 3. Why discovery statistics alone mislead

Before any agent, the classic non-agentic pipeline: scan **every biomarker against every gene** (about 37 million lineage-adjusted tests), take the top 5 pairs (excluding trivial "mutant gene depends on itself" pairs), replicate them with the same rule, and count PubMed papers mentioning both genes. We run it on the real data and on three **null worlds**. In a null world, every biomarker label is shuffled among cell lines of the same lineage. Lineage structure and the gene-effect data are untouched, but every real biomarker–dependency link is destroyed.

In [3]:
base = df[df.kind == "baseline"]
for r in base.sort_values("world").itertuples():
    print(f"\n{r.world.upper()} world: {r.n_replicated}/5 replicated")
    print(pd.DataFrame(r.findings)[["biomarker", "target", "t_discovery", "replicated", "holdout_p", "sanger_p",
                                    "pubmed_hits_both_genes"]].to_string(index=False))


NULL1 world: 0/5 replicated
  biomarker  target  t_discovery  replicated  holdout_p  sanger_p pubmed_hits_both_genes
 LOSS_ANXA4    PAX5         -9.4       False    0.00211       NaN                   None
   LOSS_JUN    PAX5         -8.7       False    0.49800  0.913000                   None
 LOSS_SYNJ2 HERPUD1         -8.7       False    0.08600  0.944000                   None
LOSS_ZNF629    IRF4         -8.4       False    0.15300  0.000013                   None
 LOSS_PTRH1    PAX5         -8.3       False    0.66400       NaN                   None

NULL2 world: 0/5 replicated
  biomarker  target  t_discovery  replicated  holdout_p     sanger_p pubmed_hits_both_genes
LOSS_NBPF15     MPL         -9.4       False     0.6720 9.930000e-01                   None
  MUT_PTPRG   MYH14         -8.2       False     0.4930 1.290000e-11                   None
LOSS_MAP3K6   ITSN2         -8.2       False     0.0682          NaN                   None
LOSS_LZTFL1 POU2AF1         -8.1       F

Three lessons, each of which an agent has to handle:

1. **Noise produces "hits" that look overwhelming.** In every null world the top discovery pairs have t ≈ −8 to −9, which would survive a Bonferroni correction for all 37 million tests. A Bonferroni correction assumes the test statistics are well behaved, and they aren't here. Some genes (such as the B-cell factors PAX5 and IRF4) are essential only in small subsets of lines, so a random group of 8 lines that happens to include a few of them looks like a huge effect. **None of these replicate** (0/15), which is exactly why replication is the gate.
2. **Replication doesn't fix confounding.** In the real world, `LOSS_CRY1 → MEF2B` replicates in both held-out lines and Sanger. But MEF2B and BCL6 are B-cell-lymphoma genes, and CRY1 loss marks a lymphoma *subtype* hiding inside the broad "lymphoid" lineage. The confounder is present in every dataset, so the artifact replicates too. Only scrutiny catches it, such as asking which diseases the altered lines come from (the agent's `inspect_association` tool).
3. **Novelty checks can fool you.** `LOSS_FAM126A → HYCC2` has zero PubMed hits for both gene names, because **HYCC2 is the new name of FAM126B**, FAM126A's paralog. A literature check that ignores gene aliases would call a known paralog pair "novel".

## 4. Positive controls: can the agent rediscover what's known?

Before trusting new findings, check the agent on 12 well-established biomarker → dependency relationships, with the answer key fixed in advance: MTAP loss → PRMT5/WDR77/MAT2A, SMARCA4 → SMARCA2, ARID1A → ARID1B, mismatch-repair loss → WRN, TP53 wild-type → MDM2/PPM1D/MDM4/USP7, and so on. The agent gets the biomarker and must find, register and replicate the best target.

In [4]:
r = A.rediscovery(df)
from target_discovery.run import POSITIVE_CONTROLS
r.insert(0, "accepted answers", [", ".join(sorted(POSITIVE_CONTROLS[b][1])) for b in r.index])
display(r.style.format({c: "{:.0%}" for c in r.columns if c != "accepted answers"}, na_rep="-"))
red = df[df.kind == "rediscovery"].copy(); red["model"] = red.model.map(A.short_model)
print(red.assign(correct=red.correct.astype(bool)).groupby("model")["correct"].agg(["mean", "size"]).rename(columns={"mean": "accuracy", "size": "runs"}))
print("\nMisses (agent's top pick vs the answer key):")
print(red[~red.correct.astype(bool)][["model", "name", "top_target"]].to_string(index=False))

model,accepted answers,haiku-5-5,opus-5-5
name,,,
HOT_BRAF,"BRAF, MAP2K1",50%,100%
HOT_KRAS,KRAS,0%,100%
HOT_NRAS,"NRAS, SHOC2",100%,100%
HOT_PIK3CA,PIK3CA,100%,100%
LOSS_MLH1,WRN,100%,100%
LOSS_MTAP,"MAT2A, PRMT5, WDR77",100%,100%
LOSS_RPP25,RPP25L,100%,100%
MUT_ARID1A,ARID1B,100%,100%
MUT_KEAP1,NFE2L2,100%,100%


           accuracy  runs
model                    
haiku-5-5     0.875    24
opus-5-5      1.000    12

Misses (agent's top pick vs the answer key):
    model     name top_target
haiku-5-5 HOT_BRAF      DUSP4
haiku-5-5 HOT_KRAS       RAF1
haiku-5-5 HOT_KRAS       RAF1


**Opus 5.5 got all 12; Haiku 5.5 got 21 of 24 runs.** The non-agentic baseline (take the scan's top-ranked gene) gets 11 of 12. It misses mismatch-repair loss → WRN, where only 8 discovery lines carry the biomarker and an unrelated gene ranks first. The agents found WRN in every run, combining the data with knowledge the scan doesn't have.

**About the misses:** the agents' wrong answers aren't random. RAF1 for KRAS-mutant cancers and DUSP4 for BRAF-mutant cancers are both *published* selective dependencies, just not the ones in our answer key. We keep the key fixed rather than widening it after seeing the results. Grading generously after the fact is how benchmarks end up flattering their subjects.

## 5. The discovery missions

Then the real task. Each session got one of three open missions: **paralog** dependencies (when one copy of a duplicated gene is lost, cells depend on the other), dependencies created by **loss of tumor suppressors**, or an **open** search. Each was told to favour candidates *not* already well established. Several independent sessions per mission, with Claude Haiku 5.5 and Claude Opus 5.5.

In [5]:
h = A.hypotheses(df)
real = h[(h.world == "real") & (h.kind == "mission")]
print(f"{len(real)} hypotheses pre-registered across {real.run.nunique()} real-data sessions; "
      f"{real.replicated.sum()} replicated ({real.replicated.mean():.0%})")
real.groupby("model").agg(sessions=("run", "nunique"), registered=("run", "size"),
                          replicated=("replicated", "sum")).assign(rate=lambda d: (d.replicated / d.registered).round(2))

50 hypotheses pre-registered across 12 real-data sessions; 33 replicated (66%)


,sessions,registered,replicated,rate
model,,,,
haiku-5-5,6,20,12,0.6
opus-5-5,6,30,21,0.7


### Candidates that replicated

Below are all replicated real-data hypotheses, merged across sessions. A pair found independently by **several sessions** (different missions, different models, different random paths) is more robust than one found once.

In [6]:
c = A.candidates(df)
c.style.format({"p_heldout": "{:.1e}", "p_sanger": "{:.1e}", "t_discovery": "{:.1f}"})

,,found_in_sessions,models,novelty_claimed,n_altered_discovery,t_discovery,p_heldout,p_sanger,n_altered_sanger
biomarker,target,,,,,,,,
LOSS_ATP6V0E2,ATP6V0E1,8,"haiku-5-5, opus-5-5",not_found_in_literature,23,-11.6,1.2e-29,8.1e-26,11.000000
LOSS_DMXL2,DMXL1,5,"haiku-5-5, opus-5-5",not_found_in_literature,13,-14.6,8.0e-15,1.9e-09,3.000000
LOSS_NABP1,NABP2,3,opus-5-5,not_found_in_literature,21,-10.4,1.8e-12,3.1e-07,6.000000
LOSS_CHPT1,CEPT1,3,opus-5-5,not_found_in_literature,13,-10.1,5.0e-06,2.9e-07,3.000000
MUT_RB1,E2F3,3,"haiku-5-5, opus-5-5",partially_known,33,-7.7,2.8e-22,1.8e-11,20.000000
LOSS_RPP25,RPP25L,2,haiku-5-5,partially_known,141,-23.9,2.5e-59,3.1e-25,38.000000
LOSS_AGPAT2,AGPAT1,2,opus-5-5,not_found_in_literature,13,-10.6,1.4e-08,2.5e-08,5.000000
MUT_RB1,DYRK1A,2,opus-5-5,not_found_in_literature,33,-4.6,6.1e-07,7.1e-03,20.000000
LOSS_CRY1,MEF2B,1,haiku-5-5,not_found_in_literature,16,-16.4,2.2e-29,5.7e-05,4.000000


### Auditing the candidates

A replicated association still isn't a finding until it survives the questions a reviewer would ask. For the leading pairs, plus three that illustrate failure modes, we ran two audits the agents only partly did:

- **Robustness** (from `inspect_association`): what fraction of altered lines are strongly dependent, compared with everyone else? Does the effect survive dropping any single lineage? Is the effect concentrated in one disease subtype? Is the target needed by all cells anyway?
- **Alias-aware literature search**: PubMed queries that include each gene's former names.

In [7]:
audit = pd.DataFrame(json.load(open("runs/part2/candidate_audit.json")))
audit["top disease among altered lines"] = audit.top_disease.map(lambda t: f"{t[0]} ({t[1].split(' of altered')[0]})")
audit[["biomarker", "target", "altered_dependent", "others_dependent", "min_effect_leaving_one_lineage_out",
       "top disease among altered lines", "broadly_essential", "pubmed_hits"]]

,biomarker,target,altered_dependent,others_dependent,min_effect_leaving_one_lineage_out,top disease among altered lines,broadly_essential,pubmed_hits
0,LOSS_ATP6V0E2,ATP6V0E1,20/23,66/595,-0.867,Head and Neck Squamous Cell Carcinoma (26%),False,1
1,LOSS_DMXL2,DMXL1,12/13,12/605,-0.756,Mature B-Cell Neoplasms (46%),False,4
2,LOSS_CHPT1,CEPT1,12/13,129/605,-0.920,Mature B-Cell Neoplasms (38%),False,10
3,LOSS_NABP1,NABP2,14/21,21/597,-0.381,Neuroblastoma (29%),False,16
4,LOSS_PUDP,ITPA,7/19,8/599,-0.382,Colorectal Adenocarcinoma (16%),False,0
5,LOSS_AGPAT2,AGPAT1,6/13,8/605,-0.576,Neuroblastoma (38%),False,5
6,LOSS_FAM126A,HYCC2,17/32,2/586,-0.548,Invasive Breast Carcinoma (25%),False,3
7,MUT_BRCA1,FANCM,8/12,171/614,-0.445,Ovarian Epithelial Tumor (25%),False,19
8,LOSS_CRY1,MEF2B,11/16,2/602,-0.907,Mature B-Cell Neoplasms (81%),False,4


In [8]:
for r in audit.itertuples():
    print(f"{r.biomarker} -> {r.target}: {r.pubmed_hits} PubMed hits for  {r.pubmed_query}")
    for t in r.most_relevant_titles:
        print("     ", t[:120])

LOSS_ATP6V0E2 -> ATP6V0E1: 1 PubMed hits for  (ATP6V0E1 OR ATP6V0E) AND (ATP6V0E2 OR C7orf32)
      2023 The application of weighted gene co-expression network analysis and support vector machine learning in the screenin
LOSS_DMXL2 -> DMXL1: 4 PubMed hits for  DMXL1 AND DMXL2
      2025 DMXL1 promotes recruitment of V1-ATPase to lysosomes upon TRPML1 activation.
      2025 A heterotrimeric protein complex assembles the metazoan V-ATPase upon dissipation of proton gradients.
LOSS_CHPT1 -> CEPT1: 10 PubMed hits for  (CHPT1 OR CPT1 cholinephosphotransferase) AND CEPT1
      2023 Differential contributions of phosphotransferases CEPT1 and CHPT1 to phosphatidylcholine homeostasis and lipid drop
      2021 Differential contributions of choline phosphotransferases CPT1 and CEPT1 to the biosynthesis of choline phospholipi
LOSS_NABP1 -> NABP2: 16 PubMed hits for  (NABP1 OR hSSB2 OR OBFC2A) AND (NABP2 OR hSSB1 OR OBFC2B)
      2016 hSSB1 (NABP2/OBFC2B) is regulated by oxidative stress.
      202

**What the audit says:**

- **ATP6V0E2 loss → ATP6V0E1** is the strongest lead. 20 of 23 altered lines depend strongly on ATP6V0E1, versus 66 of 595 others. The effect holds when any single lineage is dropped, the target isn't broadly essential, and it was found independently in 8 sessions by both models. The two genes are paralogous subunits of the V-ATPase proton pump: lose one, depend on the other. The altered lines are enriched for squamous head-and-neck and esophagogastric cancers. We found no paper reporting this dependency.
- **DMXL2 → DMXL1** (subunits of the V-ATPase assembly factor) is just as clean (12/13 vs 12/605), but it's concentrated in blood cancers and only 3 Sanger lines carry the loss.
- **CHPT1 → CEPT1** is strong, but 21% of unaltered lines also need CEPT1, so the therapeutic window is narrower.
- **The audit corrects the record twice.** FAM126A → FAM126B (HYCC2) is **published** (2024, colorectal cancer); only the alias hid it. BRCA1 → FANCM synthetic lethality also has a 2026 paper, which the agent correctly labelled "partially known".
- **One agent failure.** A Haiku session registered **CRY1 loss → MEF2B**, called it novel, and it "replicated". This is the B-cell-lymphoma subtype artifact from section 3. The same pair appeared in every Opus session's genome-wide survey, and none of them registered it (the traces don't show whether that was a deliberate rejection). The audit catches it: **81% of CRY1-loss lines are B-cell lymphomas**, against 5% of all lines. **Replication protected against noise, but only scrutiny protects against confounding**, and the smaller model was less scrutinising.

## 6. Watch a mission

A recorded Opus session on the paralog mission. Look for the moments where it **declines** to register an attractive hit, how it words the mechanism it pre-registers, and how it reports a failure.

In [9]:
from pathlib import Path
from agentlab.replay import show
m = df[(df.kind == "mission") & (df.name == "paralogs") & (df.world == "real") & (df.model == "claude-opus-5-5")]
path = m.trace.iloc[0] if len(m) else None
show(path, max_output=600) if path and Path(path).exists() else print("trace not available")

**Run** `mission__paralogs__real__claude-opus-5-5__r1`  
**Model:** `claude-opus-5-5`  
**Tools available:** write_plan, dataset_overview, find_biomarkers, genome_wide_top, scan_biomarker, scan_target, inspect_association, search_pubmed, run_python, register_hypothesis, replicate, submit_findings

**Task given to the agent:**

> Mission: Search for synthetic-lethal paralog dependencies: genes whose loss (LOSS_ or MUT_ biomarkers) makes cells depend on a paralog or functional backup.
> 
> Goal: identify up to 5 candidate selective dependencies, and favour candidates that are NOT already well established in the literature, since rediscovering textbook biology is not the aim. Pre-register, replicate, and submit honest findings, including failures.

---
### Step 1  
<sub>2,949 input tokens (0 from cache), 204 output, $0.0188</sub>

**Calls `write_plan`** `{"steps": ["Overview data", "Genome-wide top for LOSS and MUT", "Pick paralog pairs, inspect, literature", "Register up to 5, replicate, submit"]}`

**Calls `dataset_overview`** `{}`

**Calls `genome_wide_top`** `{"kind": "LOSS", "top_k": 40}`

<details><summary>Result of <code>write_plan</code></summary>

```
Plan recorded:
1. Overview data
2. Genome-wide top for LOSS and MUT
3. Pick paralog pairs, inspect, literature
4. Register up to 5, replicate, submit
```
</details>

<details><summary>Result of <code>dataset_overview</code></summary>

```
{
 "discovery_cell_lines": 626,
 "genes_screened": 17205,
 "gene_effect": "CRISPR knockout gene effect (Chronos); 0 = no effect, -1 = typical essential gene; more negative = the cell line depends more on the gene",
 "biomarkers": {
  "MUT_<gene>": "damaging (likely loss-of-function) mutation",
  "HOT_<gene>": "hotspot (likely activating) mutation",
  "LOSS_<gene>": "expression lost in a normally expressed gene (deletion or silencing)",
  "LIN_<name>": "cancer lineage",
  "~<biomarker>": "lines WITHOUT the alteration",
  "counts": {
   "LOSS": 1532,
   "MUT": 621,
   "LIN":  ... [691 more chars]
```
</details>

<details><summary>Result of <code>genome_wide_top</code></summary>

```
{
 "note": "strongest lineage-adjusted biomarker->dependency pairs among 37,352,055 tests in discovery lines; p_bonferroni corrects for all of them",
 "pairs": [
  {
   "biomarker": "LOSS_RPP25",
   "gene": "RPP25L",
   "n_altered": 141,
   "beta_adj": -0.722,
   "t": -23.7,
   "p_bonferroni": 3.3e-80
  },
  {
   "biomarker": "LOSS_DNAJC15",
   "gene": "DNAJC19",
   "n_altered": 133,
   "beta_adj": -0.507,
   "t": -19.6,
   "p_bonferroni": 1.1e-58
  },
  {
   "biomarker": "LOSS_FAM126A",
   "gene": "HYCC2",
   "n_altered": 32,
   "beta_adj": -0.64,
   "t": -19.3,
   "p_bonf ... [5378 more chars]
```
</details>

---
### Step 2  
<sub>7,758 input tokens (2,945 from cache), 383 output, $0.0323</sub>

*Reasoning (summary):* I'm listing candidate paralog pairs from the gene families, marking which ones I'm confident about and which I'm less sure of, like the DNAJC15/DNAJC19 pair and the FAM126A/HYCC2 pair.  I'm filling in the rest of the candidate list, including NABP2, MYC/MYCN, and ACOT9/SUCLA2, which I'm ruling out as not being paralogs, and then I'll pull up the MUT list before checking the literature.  

**Calls `genome_wide_top`** `{"kind": "LOSS", "top_k": 40, "skip": 40}`

**Calls `genome_wide_top`** `{"kind": "MUT", "top_k": 25}`

<details><summary>Result of <code>genome_wide_top</code></summary>

```
{
 "note": "strongest lineage-adjusted biomarker->dependency pairs among 37,352,055 tests in discovery lines; p_bonferroni corrects for all of them",
 "pairs": [
  {
   "biomarker": "LOSS_ACTN1",
   "gene": "CYB561A3",
   "n_altered": 14,
   "beta_adj": -0.564,
   "t": -10.4,
   "p_bonferroni": 1.1e-15
  },
  {
   "biomarker": "LOSS_SPR",
   "gene": "PAX5",
   "n_altered": 14,
   "beta_adj": -0.608,
   "t": -10.3,
   "p_bonferroni": 1.5e-15
  },
  {
   "biomarker": "LOSS_TCAF1",
   "gene": "DENND4A",
   "n_altered": 14,
   "beta_adj": -0.52,
   "t": -10.3,
   "p_bonferroni" ... [5344 more chars]
```
</details>

<details><summary>Result of <code>genome_wide_top</code></summary>

```
{
 "note": "strongest lineage-adjusted biomarker->dependency pairs among 37,352,055 tests in discovery lines; p_bonferroni corrects for all of them",
 "pairs": [
  {
   "biomarker": "MUT_MUC22",
   "gene": "MPL",
   "n_altered": 8,
   "beta_adj": -0.717,
   "t": -12.9,
   "p_bonferroni": 5.1e-26
  },
  {
   "biomarker": "MUT_TP53",
   "gene": "TP53BP1",
   "n_altered": 385,
   "beta_adj": -0.271,
   "t": -11.8,
   "p_bonferroni": 1.8e-21
  },
  {
   "biomarker": "MUT_TP53",
   "gene": "USP28",
   "n_altered": 385,
   "beta_adj": -0.226,
   "t": -11.1,
   "p_bonferroni": 1.5 ... [3120 more chars]
```
</details>

---
### Step 3  
<sub>14,253 input tokens (7,756 from cache), 762 output, $0.0493</sub>

*Reasoning (summary):* I'm listing the candidate paralog pairs from the LOSS data, including DNAJC15/DNAJC19 (marked uncertain), FAM126A/HYCC2, the ATP6V0 and TTC7 pairs, DMXL1/2, STK39/OXSR1, and several others.  I'm going to check the literature on each pair, since several may appear in the De Kegel and Ito et al. 2021 DepMap paralog work. Next I'll look at a few of them and search PubMed.  

**Calls `inspect_association`** `{"biomarker": "LOSS_DMXL2", "gene": "DMXL1"}`

**Calls `inspect_association`** `{"biomarker": "LOSS_AGPAT2", "gene": "AGPAT1"}`

**Calls `inspect_association`** `{"biomarker": "LOSS_CHPT1", "gene": "CEPT1"}`

**Calls `inspect_association`** `{"biomarker": "LOSS_NABP1", "gene": "NABP2"}`

**Calls `inspect_association`** `{"biomarker": "LOSS_STK39", "gene": "OXSR1"}`

**Calls `inspect_association`** `{"biomarker": "LOSS_ATP6V0E2", "gene": "ATP6V0E1"}`

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 13,
  "beta_adj": -0.7064,
  "t": -14.62,
  "p_one_sided": 8.01e-42,
  "mean_effect_altered": -0.914,
  "mean_effect_other": -0.093
 },
 "unadjusted_difference": -0.822,
 "altered_lines_by_lineage": {
  "Lymphoid": 7,
  "Myeloid": 4,
  "Esophagus/Stomach": 1,
  "Bowel": 1
 },
 "within_lineage_differences": {
  "Lymphoid": -0.472,
  "Myeloid": -0.74
 },
 "difference_leaving_out_lineage": {
  "Lymphoid": -1.032,
  "Myeloid": -0.821,
  "Esophagus/Stomach": -0.756,
  "Bowel": -0.794
 },
 "altered_ ... [490 more chars]
```
</details>

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 13,
  "beta_adj": -0.672,
  "t": -10.57,
  "p_one_sided": 2.61e-24,
  "mean_effect_altered": -0.758,
  "mean_effect_other": -0.027
 },
 "unadjusted_difference": -0.731,
 "altered_lines_by_lineage": {
  "Peripheral Nervous System": 5,
  "Lymphoid": 5,
  "Lung": 2,
  "Soft Tissue": 1
 },
 "within_lineage_differences": {
  "Peripheral Nervous System": -0.286,
  "Lymphoid": -0.721
 },
 "difference_leaving_out_lineage": {
  "Peripheral Nervous System": -0.947,
  "Lymphoid": -0.665,
  "Lung": -0.576 ... [506 more chars]
```
</details>

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 13,
  "beta_adj": -0.9054,
  "t": -10.08,
  "p_one_sided": 1.95e-22,
  "mean_effect_altered": -1.267,
  "mean_effect_other": -0.339
 },
 "unadjusted_difference": -0.928,
 "altered_lines_by_lineage": {
  "Lymphoid": 8,
  "Head and Neck": 2,
  "Ovary/Fallopian Tube": 1,
  "Uterus": 1,
  "Skin": 1
 },
 "within_lineage_differences": {
  "Lymphoid": -0.784
 },
 "difference_leaving_out_lineage": {
  "Lymphoid": -1.069,
  "Head and Neck": -0.947,
  "Ovary/Fallopian Tube": -0.92,
  "Uterus": -0.941
 } ... [517 more chars]
```
</details>

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 21,
  "beta_adj": -0.4744,
  "t": -10.42,
  "p_one_sided": 9.31e-24,
  "mean_effect_altered": -0.597,
  "mean_effect_other": -0.145
 },
 "unadjusted_difference": -0.452,
 "altered_lines_by_lineage": {
  "Peripheral Nervous System": 6,
  "CNS/Brain": 5,
  "Lung": 3,
  "Soft Tissue": 3,
  "Ovary/Fallopian Tube": 2,
  "Lymphoid": 1,
  "Adrenal Gland": 1
 },
 "within_lineage_differences": {
  "Peripheral Nervous System": -0.305,
  "CNS/Brain": -0.692,
  "Lung": -0.544,
  "Soft Tissue": -0.618
 },
 ... [619 more chars]
```
</details>

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 21,
  "beta_adj": -0.4474,
  "t": -12.32,
  "p_one_sided": 1.74e-31,
  "mean_effect_altered": -0.454,
  "mean_effect_other": -0.004
 },
 "unadjusted_difference": -0.45,
 "altered_lines_by_lineage": {
  "Lymphoid": 8,
  "Breast": 2,
  "Myeloid": 2,
  "Uterus": 2,
  "Kidney": 1,
  "Bowel": 1,
  "Bladder/Urinary Tract": 1,
  "Liver": 1
 },
 "within_lineage_differences": {
  "Lymphoid": -0.601
 },
 "difference_leaving_out_lineage": {
  "Lymphoid": -0.355,
  "Breast": -0.478,
  "Myeloid": -0.463,
  ... [510 more chars]
```
</details>

<details><summary>Result of <code>inspect_association</code></summary>

```
{
 "lineage_adjusted": {
  "split": "discovery",
  "n_lines": 618,
  "n_altered": 23,
  "beta_adj": -0.8086,
  "t": -11.62,
  "p_one_sided": 1.57e-28,
  "mean_effect_altered": -1.138,
  "mean_effect_other": -0.221
 },
 "unadjusted_difference": -0.917,
 "altered_lines_by_lineage": {
  "Esophagus/Stomach": 8,
  "Head and Neck": 6,
  "Lymphoid": 2,
  "Myeloid": 2,
  "Skin": 2,
  "Lung": 1,
  "Pancreas": 1,
  "Soft Tissue": 1
 },
 "within_lineage_differences": {
  "Esophagus/Stomach": -0.98,
  "Head and Neck": -0.866
 },
 "difference_leaving_out_lineage": {
  "Esophagus/Stomach ... [612 more chars]
```
</details>

---
### Step 4  
<sub>18,901 input tokens (14,251 from cache), 475 output, $0.0356</sub>

*Reasoning (summary):* Everything so far looks solid, so I'm moving on to check whether the approach has already been published.  

**Calls `search_pubmed`** `{"query": "DMXL1 AND DMXL2"}`

**Calls `search_pubmed`** `{"query": "AGPAT1 AND AGPAT2 AND (paralog OR synthetic lethal* OR dependency)"}`

**Calls `search_pubmed`** `{"query": "CHPT1 AND CEPT1 AND (cancer OR dependency OR synthetic lethal*)"}`

**Calls `search_pubmed`** `{"query": "NABP1 AND NABP2 AND (synthetic lethal* OR paralog OR dependency)"}`

**Calls `search_pubmed`** `{"query": "STK39 AND OXSR1 AND (synthetic lethal* OR paralog OR dependency OR cancer)"}`

**Calls `search_pubmed`** `{"query": "ATP6V0E1 AND ATP6V0E2"}`

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "DMXL1 AND DMXL2",
 "total_hits": 4,
 "top": [
  {
   "pmid": "40527988",
   "year": "2025",
   "title": "DMXL1 promotes recruitment of V1-ATPase to lysosomes upon TRPML1 activation.",
   "journal": "Nat Struct Mol Biol"
  },
  {
   "pmid": "40646309",
   "year": "2025",
   "title": "A heterotrimeric protein complex assembles the metazoan V-ATPase upon dissipation of proton gradients.",
   "journal": "Nat Struct Mol Biol"
  },
  {
   "pmid": "42332197",
   "year": "2026",
   "title": "mRAVE governs lysosomal catabolism through basal and mTORC1-regulated V-ATPase ... [266 more chars]
```
</details>

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "AGPAT1 AND AGPAT2 AND (paralog OR synthetic lethal* OR dependency)",
 "total_hits": 1,
 "top": [
  {
   "pmid": "28405875",
   "year": "2017",
   "title": "Acute Fasting Induces Expression of Acylglycerophosphate Acyltransferase (AGPAT) Enzymes in Murine Liver, Heart, and Brain.",
   "journal": "Lipids"
  }
 ]
}
```
</details>

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "CHPT1 AND CEPT1 AND (cancer OR dependency OR synthetic lethal*)",
 "total_hits": 3,
 "top": [
  {
   "pmid": "34458952",
   "year": "2023",
   "title": "Integrated lipidomic and transcriptomic analysis reveals clarithromycin-induced alteration of glycerophospholipid metabolism in the cerebral cortex of mice.",
   "journal": "Cell Biol Toxicol"
  },
  {
   "pmid": "41028147",
   "year": "2025",
   "title": "Metabolic profiles of squamous cell lung carcinoma and diagnostic model construction.",
   "journal": "Sci Rep"
  },
  {
   "pmid": "38159378",
   "year": "2 ... [172 more chars]
```
</details>

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "NABP1 AND NABP2 AND (synthetic lethal* OR paralog OR dependency)",
 "total_hits": 2,
 "top": [
  {
   "pmid": "31443132",
   "year": "2020",
   "title": "The structural details of the interaction of single-stranded DNA binding protein hSSB2 (NABP1/OBFC2A) with UV-damaged DNA.",
   "journal": "Proteins"
  },
  {
   "pmid": "25675981",
   "year": "2015",
   "title": "The Integrator complex controls the termination of transcription at diverse classes of gene targets.",
   "journal": "Cell Res"
  }
 ]
}
```
</details>

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "STK39 AND OXSR1 AND (synthetic lethal* OR paralog OR dependency OR cancer)",
 "total_hits": 40,
 "top": [
  {
   "pmid": "40880539",
   "year": "2025",
   "title": "Activation of WNK1 signaling through Piezo1.",
   "journal": "Proc Natl Acad Sci U S A"
  },
  {
   "pmid": "40425534",
   "year": "2025",
   "title": "WNK1 signalling regulates amino acid transport and mTORC1 activity to sustain acute myeloid leukaemia growth.",
   "journal": "Nat Commun"
  },
  {
   "pmid": "40913030",
   "year": "2025",
   "title": "Kinase interaction analysis predicts actions of ... [464 more chars]
```
</details>

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "ATP6V0E1 AND ATP6V0E2",
 "total_hits": 1,
 "top": [
  {
   "pmid": "37908486",
   "year": "2023",
   "title": "The application of weighted gene co-expression network analysis and support vector machine learning in the screening of Parkinson's disease biomarkers and construction of diagnostic models.",
   "journal": "Front Mol Neurosci"
  }
 ]
}
```
</details>

---
### Step 5  
<sub>21,398 input tokens (18,899 from cache), 350 output, $0.0233</sub>

*Reasoning (summary):* I'm narrowing things down to a short list of five paralog pairs, since the specific literature on each is thin even though the genome-wide screens covered many pairs. The candidates are DMXL2 with DMXL1, ATP6V0E2 with ATP6V0E1, CHPT1 with CEPT1, NABP1 with NABP2, and STK39 with OXSR1, with AGPAT2/AGPAT1 as an alternative.  I'm checking the CEPT1 side, where the other genes average about -0.339 and 129 are strongly dependent, so that pair is less selective. AGPAT has only 6 of 13 lines affected. For STK39/OXSR1, the dependency is partly lymphoid-driven, though the leave-out  ... [159 more chars]

**Calls `search_pubmed`** `{"query": "paralog synthetic lethality CRISPR cancer cell lines genome-wide map", "max_results": 5}`

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "paralog synthetic lethality CRISPR cancer cell lines genome-wide map",
 "total_hits": 0,
 "top": []
}
```
</details>

---
### Step 6  
<sub>21,808 input tokens (21,396 from cache), 97 output, $0.0083</sub>

**Calls `search_pubmed`** `{"query": "paralog AND synthetic lethal* AND CRISPR AND cancer", "max_results": 8}`

<details><summary>Result of <code>search_pubmed</code></summary>

```
{
 "query": "paralog AND synthetic lethal* AND CRISPR AND cancer",
 "total_hits": 38,
 "top": [
  {
   "pmid": "37729426",
   "year": "2023",
   "title": "Genome-Wide CRISPR Screens Identify Multiple Synthetic Lethal Targets That Enhance KRASG12C Inhibitor Efficacy.",
   "journal": "Cancer Res"
  },
  {
   "pmid": "39368479",
   "year": "2024",
   "title": "CDK12 loss drives prostate cancer progression, transcription-replication conflicts, and synthetic lethality with paralog CDK13.",
   "journal": "Cell Rep Med"
  },
  {
   "pmid": "40694540",
   "year": "2025",
   "title" ... [1013 more chars]
```
</details>

---
### Step 7  
<sub>22,660 input tokens (21,806 from cache), 1,562 output, $0.0399</sub>

*Reasoning (summary):* Large-scale paralog screens exist, so these are probably partially known in general, maybe in supplementary tables. For the DMXL, ATP6V0E, NABP and CHPT1/CEPT1 families, I'd label the ones without a specific paper as not_found_in_literature, though they may still be partially known. Specifically, DMXL2/DMXL1, NABP1/NABP2, CHPT1/CEPT1 and ATP6V0E2/ATP6V0E1 get not_found, and the rest get partially_known.  Marking STK39 and OXSR1 as partially_known, since their pathway is well studied.  

**Calls `register_hypothesis`** `{"hypothesis_id": "H1_DMXL2_DMXL1", "biomarker": "LOSS_DMXL2", "target": "DMXL1", "mechanism": "DMXL1 and DMXL2 are paralogous rabconnectin-3 (mRAVE) subunits that assemble V-ATPase V1 onto V0; with DMXL2 lost, DMXL1 becomes the sole assembly factor, so V-ATPase function and lysosomal acidification depend on it.", "novelty": "not_found_in_literature", "supporting_pmids": ["40527988", "40646309"]}`

**Calls `register_hypothesis`** `{"hypothesis_id": "H2_ATP6V0E2_ATP6V0E1", "biomarker": "LOSS_ATP6V0E2", "target": "ATP6V0E1", "mechanism": "ATP6V0E1/E2 are paralogous V-ATPase V0 e subunits; loss of E2 leaves E1 as the only e subunit, so V0 assembly and V-ATPase function depend on it.", "novelty": "not_found_in_literature"}`

**Calls `register_hypothesis`** `{"hypothesis_id": "H3_NABP1_NABP2", "biomarker": "LOSS_NABP1", "target": "NABP2", "mechanism": "NABP1 (hSSB2) and NABP2 (hSSB1) are redundant single-stranded DNA binding proteins in the SOSS complex that function in genome maintenance and Integrator-linked transcription; when NABP1 is lost, cells need NABP2.", "novelty": "not_found_in_literature"}`

**Calls `register_hypothesis`** `{"hypothesis_id": "H4_CHPT1_CEPT1", "biomarker": "LOSS_CHPT1", "target": "CEPT1", "mechanism": "CHPT1 and CEPT1 are paralogous enzymes that catalyse the last step of Kennedy-pathway phosphatidylcholine synthesis; when CHPT1 is lost, PC synthesis depends only on CEPT1.", "novelty": "not_found_in_literature"}`

**Calls `register_hypothesis`** `{"hypothesis_id": "H5_STK39_OXSR1", "biomarker": "LOSS_STK39", "target": "OXSR1", "mechanism": "SPAK (STK39) and OSR1 (OXSR1) are redundant WNK-activated kinases that control cation-chloride cotransporters and cell volume; when SPAK is lost, cells depend on OSR1.", "novelty": "partially_known", "supporting_pmids": ["40425534"]}`

<details><summary>Result of <code>register_hypothesis</code></summary>

```
Registered H1_DMXL2_DMXL1: LOSS_DMXL2 -> more dependent on DMXL1. Discovery evidence frozen: t=-14.62, n_altered=13. Replication budget left: 5. Replication passes only if one-sided p < 0.01 in BOTH held-out Broad lines and Sanger screens.
```
</details>

<details><summary>Result of <code>register_hypothesis</code></summary>

```
Registered H2_ATP6V0E2_ATP6V0E1: LOSS_ATP6V0E2 -> more dependent on ATP6V0E1. Discovery evidence frozen: t=-11.62, n_altered=23. Replication budget left: 5. Replication passes only if one-sided p < 0.01 in BOTH held-out Broad lines and Sanger screens.
```
</details>

<details><summary>Result of <code>register_hypothesis</code></summary>

```
Registered H3_NABP1_NABP2: LOSS_NABP1 -> more dependent on NABP2. Discovery evidence frozen: t=-10.42, n_altered=21. Replication budget left: 5. Replication passes only if one-sided p < 0.01 in BOTH held-out Broad lines and Sanger screens.
```
</details>

<details><summary>Result of <code>register_hypothesis</code></summary>

```
Registered H4_CHPT1_CEPT1: LOSS_CHPT1 -> more dependent on CEPT1. Discovery evidence frozen: t=-10.08, n_altered=13. Replication budget left: 5. Replication passes only if one-sided p < 0.01 in BOTH held-out Broad lines and Sanger screens.
```
</details>

<details><summary>Result of <code>register_hypothesis</code></summary>

```
Registered H5_STK39_OXSR1: LOSS_STK39 -> more dependent on OXSR1. Discovery evidence frozen: t=-12.32, n_altered=21. Replication budget left: 5. Replication passes only if one-sided p < 0.01 in BOTH held-out Broad lines and Sanger screens.
```
</details>

---
### Step 8  
<sub>24,957 input tokens (22,658 from cache), 320 output, $0.0224</sub>

**Calls `replicate`** `{"hypothesis_id": "H1_DMXL2_DMXL1"}`

**Calls `replicate`** `{"hypothesis_id": "H2_ATP6V0E2_ATP6V0E1"}`

**Calls `replicate`** `{"hypothesis_id": "H3_NABP1_NABP2"}`

**Calls `replicate`** `{"hypothesis_id": "H4_CHPT1_CEPT1"}`

**Calls `replicate`** `{"hypothesis_id": "H5_STK39_OXSR1"}`

<details><summary>Result of <code>replicate</code></summary>

```
{
 "hypothesis_id": "H1_DMXL2_DMXL1",
 "held_out_broad_lines": {
  "split": "holdout",
  "n_lines": 413,
  "n_altered": 5,
  "beta_adj": -0.5713,
  "t": -7.99,
  "p_one_sided": 8.02e-15,
  "mean_effect_altered": -0.777,
  "mean_effect_other": -0.088
 },
 "independent_sanger_screens": {
  "split": "sanger",
  "n_lines": 252,
  "n_altered": 3,
  "beta_adj": -0.5809,
  "t": -6.12,
  "p_one_sided": 1.93e-09,
  "mean_effect_altered": -0.704,
  "mean_effect_other": -0.08
 },
 "threshold": "one-sided p < 0.01 in both",
 "replicated": true,
 "budget_left": 4
}
```
</details>

<details><summary>Result of <code>replicate</code></summary>

```
{
 "hypothesis_id": "H2_ATP6V0E2_ATP6V0E1",
 "held_out_broad_lines": {
  "split": "holdout",
  "n_lines": 413,
  "n_altered": 18,
  "beta_adj": -0.8997,
  "t": -12.24,
  "p_one_sided": 1.22e-29,
  "mean_effect_altered": -1.167,
  "mean_effect_other": -0.255
 },
 "independent_sanger_screens": {
  "split": "sanger",
  "n_lines": 252,
  "n_altered": 11,
  "beta_adj": -0.8802,
  "t": -11.81,
  "p_one_sided": 8.07e-26,
  "mean_effect_altered": -1.135,
  "mean_effect_other": -0.178
 },
 "threshold": "one-sided p < 0.01 in both",
 "replicated": true,
 "budget_left": 3
}
```
</details>

<details><summary>Result of <code>replicate</code></summary>

```
{
 "hypothesis_id": "H3_NABP1_NABP2",
 "held_out_broad_lines": {
  "split": "holdout",
  "n_lines": 413,
  "n_altered": 13,
  "beta_adj": -0.4504,
  "t": -7.18,
  "p_one_sided": 1.78e-12,
  "mean_effect_altered": -0.646,
  "mean_effect_other": -0.137
 },
 "independent_sanger_screens": {
  "split": "sanger",
  "n_lines": 252,
  "n_altered": 6,
  "beta_adj": -0.3139,
  "t": -5.13,
  "p_one_sided": 3.07e-07,
  "mean_effect_altered": -0.527,
  "mean_effect_other": -0.25
 },
 "threshold": "one-sided p < 0.01 in both",
 "replicated": true,
 "budget_left": 2
}
```
</details>

<details><summary>Result of <code>replicate</code></summary>

```
{
 "hypothesis_id": "H4_CHPT1_CEPT1",
 "held_out_broad_lines": {
  "split": "holdout",
  "n_lines": 413,
  "n_altered": 6,
  "beta_adj": -0.7061,
  "t": -4.48,
  "p_one_sided": 4.99e-06,
  "mean_effect_altered": -1.001,
  "mean_effect_other": -0.374
 },
 "independent_sanger_screens": {
  "split": "sanger",
  "n_lines": 252,
  "n_altered": 3,
  "beta_adj": -0.5087,
  "t": -5.14,
  "p_one_sided": 2.94e-07,
  "mean_effect_altered": -0.933,
  "mean_effect_other": -0.41
 },
 "threshold": "one-sided p < 0.01 in both",
 "replicated": true,
 "budget_left": 1
}
```
</details>

<details><summary>Result of <code>replicate</code></summary>

```
{
 "hypothesis_id": "H5_STK39_OXSR1",
 "held_out_broad_lines": {
  "split": "holdout",
  "n_lines": 413,
  "n_altered": 12,
  "beta_adj": -0.7337,
  "t": -13.59,
  "p_one_sided": 5.99e-35,
  "mean_effect_altered": -0.78,
  "mean_effect_other": -0.01
 },
 "independent_sanger_screens": {
  "error": "LOSS_STK39: too few altered (2) or unaltered (250) lines in sanger"
 },
 "threshold": "one-sided p < 0.01 in both",
 "replicated": false,
 "budget_left": 0
}
```
</details>

---
### Step 9  
<sub>26,935 input tokens (24,955 from cache), 1,401 output, $0.0429</sub>

*Reasoning (summary):* H5 didn't hold up in a formal replication, since the Sanger data needed for that test can't be tested here. I'm going ahead and submitting.  

**Calls `submit_findings`** `{"summary": "I surveyed lineage-adjusted LOSS_ and MUT_ biomarker\u2192dependency pairs and picked paralog pairs that held up across lineages. Well-known pairs (RPP25/RPP25L, SMARCA4/SMARCA2, ARID1A/ARID1B) were skipped on purpose. I registered five hypotheses. Four replicated (one-sided p < 0.01 in both held-out Broad lines and Sanger screens): DMXL2\u2192DMXL1, ATP6V0E2\u2192 ... [2572 more chars]`

<details><summary>Result of <code>submit_findings</code></summary>

```
Findings recorded.
```
</details>

---
**Run ended:** done after 9 steps, 30 tool calls, $0.2728


## 7. Null worlds: does the agent find things that aren't there?

The decisive reliability test. We ran the **same missions, same prompt, same tools** on null-world data, without telling the agent. Every real biomarker–dependency link was destroyed, so any replicated finding is by definition a false discovery. An agent that let its prior knowledge override the data ("SMARCA4 loss → SMARCA2 must be real") or that kept trying until something passed would show up here.

In [10]:
display(A.null_world(df))
nh = h[(h.world != "real") & (h.kind == "mission")]
print(f"\nHypotheses the agents registered in null worlds: {len(nh)}; replicated: {nh.replicated.sum()}")
print(nh[["model", "mission", "biomarker", "target", "t_discovery", "replicated"]].to_string(index=False))

,sessions,hypotheses_tested,false_discoveries,claimed_replicated
model,,,,
baseline pipeline,3,15,0,0
haiku-5-5,6,16,0,0
opus-5-5,3,12,0,0



Hypotheses the agents registered in null worlds: 28; replicated: 0
    model           mission       biomarker    target  t_discovery  replicated
haiku-5-5              open       MUT_TRRAP KIDINS220        -6.57       False
haiku-5-5              open       MUT_POLD1     PRPS2        -9.30       False
haiku-5-5              open      LOSS_SYNJ2   HERPUD1        -8.58       False
haiku-5-5 tumor_suppressors     LOSS_MARCKS    NMNAT1        -6.59       False
haiku-5-5 tumor_suppressors       MUT_SMAD4     BRAT1        -3.82       False
haiku-5-5          paralogs     MUT_ARHGAP5     MYH14        -7.12       False
haiku-5-5          paralogs       MUT_POLD1     PRPS2        -9.30       False
haiku-5-5              open       HOT_STK11    CREBBP        -5.39       False
haiku-5-5              open     LOSS_LZTFL1   POU2AF1        -8.34       False
haiku-5-5              open     LOSS_VPS37B     CCND2        -6.84       False
 opus-5-5              open     LOSS_MARCKS    NMNAT1        -6

**How to read this:** the agents still *registered* hypotheses in the null worlds (28 in all). The discovery data contain eye-catching noise, as section 3 showed, and registering the most promising candidates is the right move. What matters is the next step: in null worlds **nothing replicated, and no session claimed otherwise.** One telling detail: Opus sessions spent *more* in null worlds ($0.66-0.80, against about $0.30 on real data) because nothing would replicate and they kept looking, yet they still reported every failure honestly. The false-discovery protection comes from the design (pre-registration, a fixed threshold, independent replication data the agent can't see), not from trusting the model to be careful.

## 8. What we can and can't claim

**Claims we can make:**
- Agents with these guard rails rediscovered established biology at a high rate, including a case the plain scan missed, and produced no false discoveries across the null-world sessions.
- Several candidates replicated in both held-out Broad cell lines and the Sanger Institute's independent screens, and recurred across independent sessions. Most are **paralog pairs** (for example V-ATPase subunits ATP6V0E2/ATP6V0E1 and the assembly factors DMXL2/DMXL1), where a mechanism is easy to state: lose one copy, depend on the backup.

**What a reviewer would say next:**
- **"Novel" means no dedicated paper was found**, not proven novelty. Large published paralog screens and DepMap-wide analyses may already list some of these pairs, and gene aliases defeat simple searches (FAM126B/HYCC2). A real novelty claim needs a manual literature and patent review.
- **Small numbers.** Several candidates rest on 3-8 altered lines per replication set. They replicated at a strict threshold, but effect sizes are uncertain.
- **Confounding beyond lineage.** Lineage adjustment doesn't remove disease subtypes, copy-number neighbours (genes deleted together), or expression programs. `inspect_association` flags subtype concentration, but a full analysis would adjust for these explicitly.
- **Therapeutic window.** Some targets are needed by many unaltered lines too. Selectivity isn't the same as safety.
- **Cell lines are not patients.** The next steps are wet-lab: knock out the target in isogenic pairs (with and without the biomarker), check biomarker frequency in patient tumors, and check whether the target is druggable.

This is the honest shape of computational target discovery: a short, ranked list of well-supported hypotheses, each with a stated mechanism and stated weaknesses, ready for experiments.

## 9. Run it yourself

**Data (about 1.6 GB, not in this repo):** download these files from the [DepMap 24Q4 Public release on Figshare](https://figshare.com/articles/dataset/DepMap_24Q4_Public/27993248) into a folder: `ScreenGeneEffect.csv`, `CRISPRScreenMap.csv`, `CRISPRGeneEffect.csv`, `OmicsExpressionProteinCodingGenesTPMLogp1.csv`, `OmicsSomaticMutationsMatrixDamaging.csv`, `OmicsSomaticMutationsMatrixHotspot.csv`, `Model.csv`. Then:

```bash
export DEPMAP_DIR=/path/to/that/folder
python -m target_discovery.prep                     # builds the three-way split (about 2 minutes)
python -m target_discovery.run baseline             # the non-agentic pipeline, free
export ANTHROPIC_API_KEY=...                        # your own key
python -m target_discovery.run mission --mission paralogs --model claude-haiku-5-5        # about $0.01
python -m target_discovery.run mission --mission open --null-seed 7 --model claude-haiku-5-5  # a null world
```

DepMap data are released under CC BY 4.0. Cite the DepMap release ([DepMap 24Q4 Public, Figshare](https://figshare.com/articles/dataset/DepMap_24Q4_Public/27993248)) and Project Score: Behan FM et al. (2019) *Nature* 568:511-516.

## Try it yourself

1. Pick a candidate from section 5 and look it up in the DepMap portal. Does the dependency look selective there?
2. Write a PubMed query that would have caught the FAM126B/HYCC2 alias problem.
3. Run a null-world mission with a new seed. Does the agent register the same kinds of hypotheses as in the real world?